# Bash Scripting Fundamentals

**Goal:** general-purpose reference for reading and writing bash scripts — syntax, control flow, functions, text processing, and the shell idioms you'll run into constantly (in scripts, CI/CD, Docker, git hooks, etc.).

**How this notebook runs:** the kernel is Python, but every code cell starts with the `%%bash` magic, which hands the entire cell to `/bin/bash` as a script and shows you the real output. This means you don't need a separate bash kernel — it works in any standard Jupyter/Cursor notebook.

**Format:** same reading-first philosophy as the rest of this project — real explanation grounded in actual shell behavior, plus a runnable, verified example per topic. No TODO placeholders.

Sources: [GNU Bash Reference Manual](https://www.gnu.org/software/bash/manual/bash.html) · [Bash Guide (Greg's Wiki)](https://mywiki.wooledge.org/BashGuide) — cross-checked by actually executing every cell.

---


## 1 Running a script: shebang, permissions, execution

Every bash *script* (a `.sh` file) normally starts with a **shebang** line: `#!/bin/bash` (or `#!/usr/bin/env bash` for portability). This tells the OS which interpreter to run the file with. To run a script directly (`./script.sh`) it needs the executable bit: `chmod +x script.sh`. Without the executable bit, you can still run it explicitly with `bash script.sh`.

Inside a notebook cell, `%%bash` does the same job as the shebang — everything below it runs as a bash script.


In [ ]:
%%bash
echo "Hello from bash"
echo "Current shell: $0"
echo "Bash version: $BASH_VERSION"


Hello from bash
Current shell: bash
Bash version: 5.1.16(1)-release


## 2 Variables, quoting, and command substitution

Variables are assigned with **no spaces** around `=` (`name=value`), and read with a `$` prefix (`$name` or `${name}`). Three quoting styles matter:

- **Double quotes `"..."`** — variables and command substitutions are expanded inside.
- **Single quotes `'...'`** — everything is literal, nothing is expanded.
- **No quotes** — the shell performs word-splitting and glob expansion on the result, which is usually *not* what you want (always quote variables unless you specifically need splitting).

**Command substitution** `$(command)` runs a command and substitutes its output as a string — this is how you capture a command's result into a variable. (The backtick form `` `command` `` is older and equivalent but harder to nest.)


In [ ]:
%%bash
name="Javier"
greeting="Hello, $name!"          # double quotes: $name expands
literal='Hello, $name!'           # single quotes: literal, no expansion
echo "$greeting"
echo "$literal"

today=$(date +%Y-%m-%d)            # command substitution
echo "Today is: $today"

# unquoted vs quoted with spaces -- the classic gotcha
file="my file.txt"
touch "$file"
ls -la "$file"      # correct: quoted, treated as ONE argument
rm "$file"


Hello, Javier!
Hello, $name!


Today is: 2026-07-17


-rw-r--r-- 1 festive-peaceful-johnson festive-peaceful-johnson 0 Jul 17 09:27 my file.txt


## 3 Script arguments and special variables

When a script runs, the shell exposes its arguments and metadata through special variables: `$0` is the script name itself, `$1`, `$2`, ... are positional arguments, `$#` is the argument count, `$@` expands to all arguments as separate words (almost always what you want — quote it as `"$@"`), and `$?` holds the exit status of the *last* command (0 = success, non-zero = failure). `read` lets a script prompt for interactive input.


In [ ]:
%%bash
set -- alpha beta gamma     # simulate script arguments $1 $2 $3 (set -- sets positional params)
echo "Script name (\$0): $0"
echo "First arg (\$1): $1"
echo "Arg count (\$#): $#"
echo "All args (\$@): $@"

true   # a command that always succeeds
echo "Exit status of 'true': $?"
false  # a command that always fails
echo "Exit status of 'false': $?"


Script name ($0): bash


First arg ($1): alpha
Arg count ($#): 3


All args ($@): alpha beta gamma
Exit status of 'true': 0
Exit status of 'false': 1


## 4 Conditionals: if / elif / else and test expressions

`if` checks the **exit status** of a command — 0 means true, anything else means false. The `test` command (or its bracket form `[ expr ]`) is the most common thing to test: `[[ expr ]]` is bash's improved version (supports `&&`/`||`/`==` pattern matching directly, no word-splitting surprises) and is generally preferred over the older POSIX `[ ]`.

Common comparisons: strings use `==`/`!=` (or `=`), numbers use `-eq -ne -lt -le -gt -ge` (NOT `<` `>` inside `[ ]`), and **file test operators** check the filesystem: `-f` (regular file exists), `-d` (directory exists), `-e` (exists, any type), `-r`/`-w`/`-x` (readable/writable/executable).


In [4]:
%%bash
num=7

if [[ $num -gt 10 ]]; then
    echo "big"
elif [[ $num -gt 5 ]]; then
    echo "medium"
else
    echo "small"
fi

# string comparison
name="Ada"
if [[ "$name" == "Ada" ]]; then
    echo "Hello Ada"
fi

# file test operators
touch /tmp/demo_bash_file.txt
if [[ -f /tmp/demo_bash_file.txt ]]; then
    echo "file exists"
fi
if [[ -d /tmp ]]; then
    echo "/tmp is a directory"
fi
rm /tmp/demo_bash_file.txt


medium


Hello Ada


file exists
/tmp is a directory


## 5 Loops: for, while, until, break/continue

`for var in list; do ... done` iterates over a list of words (or `$(command)` output, or a glob like `*.txt`). `for ((i=0; i<n; i++))` is the C-style numeric form. `while condition; do ... done` repeats while the condition is true; `until` repeats while it's *false* (i.e. until it becomes true). `break` exits the loop early, `continue` skips to the next iteration.


In [5]:
%%bash
for fruit in apple banana cherry; do
    echo "Fruit: $fruit"
done

for ((i = 0; i < 5; i++)); do
    if (( i == 3 )); then
        continue
    fi
    echo "i = $i"
done

count=0
while (( count < 3 )); do
    echo "while count: $count"
    (( count++ ))
done

n=0
until (( n >= 2 )); do
    echo "until n: $n"
    (( n++ ))
done


Fruit: apple
Fruit: banana
Fruit: cherry


i = 0
i = 1
i = 2
i = 4
while count: 0
while count: 1
while count: 2
until n: 0
until n: 1


## 6 Functions

Functions are defined with `name() { ... }` (or `function name { ... }`). Bash functions don't "return" values the way Python does — `return N` sets the function's **exit status** (0-255, for success/failure signaling, checked via `$?`), and if you need to return *data*, the convention is to `echo` it and capture the output with `$(function_name ...)`. Arguments inside a function are `$1`, `$2`, ... just like script arguments. Use `local` to keep a variable scoped to the function instead of leaking into the global shell.


In [6]:
%%bash
greet() {
    local person="$1"      # local: scoped to this function only
    echo "Hello, $person!"
}
greet "Javier"

add() {
    local sum=$(( $1 + $2 ))
    echo "$sum"             # "returning" data via echo
}
result=$(add 3 4)           # captured via command substitution
echo "3 + 4 = $result"

is_even() {
    if (( $1 % 2 == 0 )); then
        return 0    # 0 = success/true
    else
        return 1    # non-zero = failure/false
    fi
}
if is_even 10; then
    echo "10 is even"
fi


Hello, Javier!


3 + 4 = 7


10 is even


## 7 Arrays

Bash supports indexed arrays: `arr=(a b c)`. Access an element with `${arr[i]}` (0-indexed), all elements with `${arr[@]}`, and length with `${#arr[@]}`. Append with `arr+=(d)`. Associative arrays (`declare -A`) work like dictionaries — key/value pairs — but must be declared explicitly first.


In [7]:
%%bash
colors=(red green blue)
echo "First: ${colors[0]}"
echo "All: ${colors[@]}"
echo "Count: ${#colors[@]}"

colors+=(yellow)
echo "After append: ${colors[@]}"

for c in "${colors[@]}"; do
    echo "color -> $c"
done

declare -A capitals
capitals[colombia]="Bogota"
capitals[france]="Paris"
echo "Capital of France: ${capitals[france]}"
for country in "${!capitals[@]}"; do    # ! gives the KEYS of an associative array
    echo "$country -> ${capitals[$country]}"
done


First: red
All: red green blue


Count: 3
After append: red green blue yellow
color -> red
color -> green
color -> blue
color -> yell

ow
Capital of France: Paris
france -> Paris
colombia -> Bogota


## 8 String manipulation

Bash has built-in string operations without needing external tools: `${#str}` gives length; `${str:start:len}` extracts a substring; `${str/old/new}` replaces the first match, `${str//old/new}` replaces all matches; `${str^^}`/`${str,,}` uppercase/lowercase the whole string; `${str#prefix}`/`${str%suffix}` strip a prefix/suffix (useful for stripping file extensions).


In [8]:
%%bash
str="Hello, World!"
echo "Length: ${#str}"
echo "Substring (0,5): ${str:0:5}"
echo "Replace first: ${str/World/Bash}"
echo "Uppercase: ${str^^}"
echo "Lowercase: ${str,,}"

filename="report.final.txt"
echo "Strip suffix: ${filename%.txt}"          # remove shortest match from the end
echo "Strip prefix: ${filename#report.}"        # remove shortest match from the start


Length: 13
Substring (0,5): Hello


Replace first: Hello, Bash!
Uppercase: HELLO, WORLD!
Lowercase: hello, world!
Strip suffix: report.f

inal
Strip prefix: final.txt


## 9 Pipes and redirection

`|` sends one command's **stdout** as the next command's **stdin** — this is how you chain small tools into a pipeline. Redirection controls where a command's streams go: `>` overwrites a file with stdout, `>>` appends, `<` feeds a file in as stdin, `2>` redirects **stderr** specifically, and `&>` (or `> file 2>&1`) redirects both stdout and stderr together. `&&` runs the next command only if the previous one succeeded; `||` runs it only if the previous one failed.


In [9]:
%%bash
echo -e "banana\napple\ncherry" | sort           # pipe: sort's stdin comes from echo's stdout
echo -e "banana\napple\ncherry" | sort | head -n 2   # chained pipeline

echo "line 1" > /tmp/demo_pipes.txt      # overwrite
echo "line 2" >> /tmp/demo_pipes.txt     # append
cat /tmp/demo_pipes.txt
rm /tmp/demo_pipes.txt

mkdir -p /tmp/demo_dir && echo "mkdir succeeded, so this runs"
rmdir /tmp/nonexistent_dir 2>/dev/null || echo "rmdir failed, so this ran instead"
rmdir /tmp/demo_dir


apple
banana
cherry


apple
banana


line 1
line 2


mkdir succeeded, so this runs


rmdir failed, so this ran instead


## 10 Common text-processing utilities: grep, sed, cut, sort, wc, awk

These small tools are the backbone of shell text processing, almost always used via pipes:

- **`grep pattern`** — filters lines matching a pattern (`-i` case-insensitive, `-v` invert match, `-n` show line numbers, `-E` for extended regex).
- **`sed 's/old/new/'`** — stream editor; the classic use is find-and-replace (`s/old/new/g` for all occurrences on a line, not just the first).
- **`cut -d, -f2`** — extracts a field from delimited text (here: comma-delimited, 2nd field).
- **`sort`** / **`sort -n`** / **`sort -r`** — sorts lines (alphabetically by default, `-n` numeric, `-r` reverse).
- **`wc -l`** — counts lines (`-w` words, `-c` bytes).
- **`awk '{print $1}'`** — pattern-scanning/processing language; the simplest use splits each line into whitespace-separated fields (`$1`, `$2`, ...) and lets you print/transform them.


In [10]:
%%bash
data="apple,3
banana,7
cherry,1"

echo "$data" | grep "an"                      # lines containing 'an'
echo "---"
echo "$data" | sed 's/,/ -> /'                 # replace comma with arrow
echo "---"
echo "$data" | cut -d, -f1                     # just the fruit names (field 1)
echo "---"
echo "$data" | sort -t, -k2 -n                 # numeric sort by 2nd field
echo "---"
echo "$data" | wc -l                           # line count
echo "---"
echo "$data" | awk -F, '{print $1, "qty:", $2}'   # -F, sets the field separator to comma


banana,7


---


apple -> 3
banana -> 7
cherry -> 1


---


apple
banana
cherry


---


cherry,1
apple,3
banana,7


---


3


---


apple qty: 3
banana qty: 7
cherry qty: 1


## 11 Exit codes and error handling

Every command returns an **exit code**: `0` for success, `1`-`255` for various failure reasons — this is the mechanism `if`, `&&`, `||`, and `$?` all rely on. For scripts (not just one-off cells), two settings make error handling much safer:

- **`set -e`** — the script exits immediately if any command fails (instead of silently continuing with a bad state).
- **`set -u`** — treat use of an undefined variable as an error (catches typos in variable names).

`trap 'command' SIGNAL` lets a script run cleanup code when it exits or receives a signal — commonly `trap cleanup EXIT` to guarantee temp files get removed even if the script fails partway through.


In [11]:
%%bash
set -u
# echo "$undefined_var"   # would abort the script here because of set -u -- commented out so this cell keeps running

cleanup() {
    echo "cleanup ran"
    rm -f /tmp/demo_trap.txt
}
trap cleanup EXIT     # runs 'cleanup' automatically when this script/cell finishes

touch /tmp/demo_trap.txt
echo "created temp file, doing work..."
echo "script ending now (watch cleanup fire automatically below)"


created temp file, doing work...


script ending now (watch cleanup fire automatically below)
cleanup ran
